Aprendizado de variedades (Manifold Learning) com fluxo de clusterização distribuída por tipo de característica é mais informativo do que o UMAP para conjuntos de dados clínicos tabulares

Importando as Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
import math
import umap.umap_ as umap
%config InlineBackend.figure_format = 'svg'

In [ ]:
from cluster_val import *

Importando os dados

In [ ]:
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
data_with_target=pd.read_csv('survey lung cancer.csv')

Pré-processamento dos dados

In [ ]:
data_with_target.isna().sum() #Detecta valores faltando por coluna

In [ ]:
#Transforma variável categórica (F/M) em numérica (1/0) -> label encoding manual
gender_mod= {'GENDER': {'F':1,'M':0}}
data_with_target.replace(gender_mod,inplace=True)
data_with_target['GENDER']

In [ ]:
np.random.seed(42)
data_with_target=data_with_target.sample(frac=1) #Shuffle the data set
np.random.seed(42)
i=[x for x in range(309)]

data_with_target.set_index(pd.Series(i), inplace=True)

In [ ]:
data_with_target['LUNG_CANCER']

In [ ]:
data=data_with_target.drop(['LUNG_CANCER'],axis=1)

In [ ]:
data.shape

UMAP nos dados originais

In [ ]:
from fdc.fdc import feature_clustering

umap_emb=feature_clustering(15,0.1,'euclidean',data,True)

Silhouette_score e Dunn index para clusters UMAP extraidos usando k-means

In [ ]:
from fdc.clustering import Clustering

umap_clustering=Clustering(umap_emb,umap_emb,True)
umap_cluster_list,umap_cluster_counts=umap_clustering.K_means(3)

In [ ]:
from sklearn import metrics
from sklearn.metrics import pairwise_distances
from sklearn.metrics import silhouette_score

silhouette_score(umap_emb, umap_cluster_list, metric='euclidean')

Visualizing Silhouette score (Pode-se escolher o número de clusters baseado no score)

In [ ]:
Silhouette_visual(umap_emb)

Elbow plot para o umap_embedding

In [ ]:
elbow_plot(umap_emb)

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list))

Silhouette_score e Dunn index para clusters UMAP extraidos usando clustering Aglomerativo

In [ ]:
umap_cluster_list_agglo,umap_cluster_counts_agglo=umap_clustering.Agglomerative(3,'euclidean','ward')

In [ ]:
silhouette_score(umap_emb, umap_cluster_list_agglo, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list_agglo))

Silhouette_score e Dunn index para clusters UMAP extraidos usando clustering do DBSCAN

In [ ]:
umap_cluster_list_dbscan,umap_cluster_counts_dbscan=umap_clustering.DBSCAN(0.5,20)

In [ ]:
#Remove os indices considerados ruídos pelo algoritmo
non_noise_indices= np.where(np.array(umap_cluster_list_dbscan)!=-1)
umap_emb= umap_emb.iloc[non_noise_indices]
#FDC_emb_low= FDC_emb_low.iloc[non_noise_indices]
umap_cluster_list_dbscan= np.array(umap_cluster_list_dbscan)[non_noise_indices]

In [ ]:
silhouette_score(umap_emb, umap_cluster_list_dbscan, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list_dbscan))

Dividindo as variaveis
- cont_list = continuas
- ord_list = ordinais
-  nom_list  = nominais

In [ ]:
cont_list=['AGE']

ord_list=['SMOKING','GENDER','CHRONIC DISEASE','ALCOHOL CONSUMING','SHORTNESS OF BREATH']

nom_list=['YELLOW_FINGERS','ANXIETY','PEER_PRESSURE','WHEEZING','COUGHING','SWALLOWING DIFFICULTY','CHEST PAIN','FATIGUE ','ALLERGY ']

In [ ]:
len(ord_list)

In [ ]:
len(nom_list)

In [ ]:
len(cont_list)

Usando FDC nos dados originais

In [ ]:
from fdc.fdc import FDC, Clustering
from fdc.fdc import canberra_modified
modified_can = canberra_modified

In [ ]:
fdc = FDC(
    clustering_cont=Clustering('euclidean', 15, 0.1, max_components=1)  # usa distância euclidiana para variáveis contínuas
    , clustering_ord=Clustering('canberra', 15, 0.1)                    # usa distância de Canberra para variáveis ordinais
    , clustering_nom=Clustering('hamming', 15, 0.1)                     # usa distância de Hamming para variáveis nominais (categóricas sem ordem)
    , visual=True              # ativa visualização automática dos clusters e embeddings
    , use_pandas_output=True   # faz o retorno ser em formato de DataFrame do pandas
    , with_2d_embedding=True   # gera uma projeção em 2D para visualização dos dados
)

fdc.selectFeatures(
    continueous=cont_list,
    nomial=nom_list,
    ordinal=ord_list
) # informa ao modelo quais variáveis são contínuas, nominais e ordinais


FDC_emb_high, FDC_emb_low = fdc.normalize(
    data,
    n_neighbors=30, # número de vizinhos usados na construção do grafo (mais alto → clusters mais globais)
    min_dist=0.001, # controla o quão próximos os pontos podem ficar no embedding, valor muito baixo → clusters muito compactos e bem definidos
    cont_list=cont_list,
    nom_list=nom_list,
    ord_list=ord_list,
    with_2d_embedding=True, #Gera duas representações, uma de alta e outra de baixa dimensão
    visual=True # exibe visualizações automáticas dos clusters e embeddings
)

Silhouette_score e Dunn index para clusters FDC(Dimensão intermediária) extraidos usando clustering do K-means

In [ ]:
from fdc.clustering import Clustering

clustering=Clustering(FDC_emb_high,FDC_emb_low,True)
cluster_list,cluster_counts=clustering.K_means(6)

In [ ]:
FDC_emb_high['Cluster'] = cluster_list

In [ ]:
silhouette_score(FDC_emb_high, cluster_list, metric='euclidean')

In [ ]:
Silhouette_visual(FDC_emb_high)

In [ ]:
elbow_plot(FDC_emb_high)

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_high,cluster_list))

Silhouette_score e Dunn index para clusters FDC(Dimensão intermediária) extraidos usando clustering Aglomerativo

In [ ]:
cluster_list_agglo,cluster_counts_agglo=clustering.Agglomerative(6,'euclidean','ward')

In [ ]:
FDC_emb_high['Cluster'] = cluster_list_agglo

In [ ]:
silhouette_score(FDC_emb_high, cluster_list_agglo, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_high,cluster_list_agglo))

Silhouette_score e Dunn index para clusters FDC(Dimensão intermediária) extraidos usando clustering do DBSCAN

In [ ]:
cluster_list_dbscan,cluster_counts_dbscan=clustering.DBSCAN(1.2,20)

In [ ]:
cluster_counts_dbscan

In [ ]:
FDC_emb_high['Cluster'] = cluster_list_dbscan

In [ ]:
#Remove indices considerados ruídos
non_noise_indices= np.where(np.array(cluster_list_dbscan)!=-1)
FDC_emb_high= FDC_emb_high.iloc[non_noise_indices]
FDC_emb_low= FDC_emb_low.iloc[non_noise_indices]
cluster_list_dbscan= np.array(cluster_list_dbscan)[non_noise_indices]

In [ ]:
silhouette_score(FDC_emb_high, cluster_list_dbscan, metric='euclidean') #média de separação ponto a ponto

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_high,cluster_list_dbscan)) #pior caso de separação vs pior dispersão